# Toxicité moléculaire avec GNN (version améliorée)

Ce notebook montre un modèle GNN amélioré avec DeepChem.

In [5]:
import deepchem as dc
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

## Chargement dataset Tox21

In [6]:
# 1. On crée le convertisseur en graphes
featurizer = dc.feat.ConvMolFeaturizer()

# 2. On charge Tox21 en forçant l'utilisation de ce convertisseur
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer=featurizer)
train_dataset, valid_dataset, test_dataset = datasets


## Modèle GNN amélioré

In [7]:
model = dc.models.GraphConvModel(
    n_tasks=len(tasks),
    mode='classification',
    dropout=0.3,
    batch_size=64,
    learning_rate=5e-4
)

## Entraînement

In [8]:
model.fit(train_dataset, nb_epoch=50)

0.7098570251464844

## Évaluation

In [9]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)
print(model.evaluate(test_dataset, [metric]))

{'roc_auc_score': 0.6969994163309748}


## Test sur molécules

In [11]:
# Prédictions brutes
predictions = model.predict(dataset)

noms_molecules = ["Paracétamol", "Valdécoxib"]

for i, nom in enumerate(noms_molecules):
    print(f"\n=== PROFIL DE TOXICITÉ : {nom} ===")
    
    # Pour chaque molécule, on regarde ses 12 scores
    for index, nom_cible in enumerate(tasks):
        # On regarde la probabilité d'être toxique (indice 1)
        probabilite = predictions[i][index][1] 
        
        if probabilite > 0.5:
            print(f"⚠️ DANGER sur {nom_cible:10s} : {probabilite*100:.1f}%")
        else:
            print(f"  Sûr    sur {nom_cible:10s} : {probabilite*100:.1f}%")



=== PROFIL DE TOXICITÉ : Paracétamol ===
  Sûr    sur NR-AR      : 27.2%
  Sûr    sur NR-AR-LBD  : 9.5%
⚠️ DANGER sur NR-AhR     : 77.0%
  Sûr    sur NR-Aromatase : 13.7%
⚠️ DANGER sur NR-ER      : 71.8%
⚠️ DANGER sur NR-ER-LBD  : 78.0%
⚠️ DANGER sur NR-PPAR-gamma : 63.6%
⚠️ DANGER sur SR-ARE     : 56.0%
⚠️ DANGER sur SR-ATAD5   : 72.5%
  Sûr    sur SR-HSE     : 36.2%
⚠️ DANGER sur SR-MMP     : 61.4%
  Sûr    sur SR-p53     : 46.5%

=== PROFIL DE TOXICITÉ : Valdécoxib ===
  Sûr    sur NR-AR      : 22.5%
  Sûr    sur NR-AR-LBD  : 38.9%
⚠️ DANGER sur NR-AhR     : 82.4%
⚠️ DANGER sur NR-Aromatase : 78.6%
  Sûr    sur NR-ER      : 37.4%
  Sûr    sur NR-ER-LBD  : 30.2%
⚠️ DANGER sur NR-PPAR-gamma : 71.5%
  Sûr    sur SR-ARE     : 44.6%
  Sûr    sur SR-ATAD5   : 27.5%
  Sûr    sur SR-HSE     : 20.0%
⚠️ DANGER sur SR-MMP     : 77.9%
  Sûr    sur SR-p53     : 25.5%
